In [1]:
!uv sync

Resolved 17 packages in 22ms
Audited 16 packages in 4ms


In [2]:
import jax
import jax.numpy as jnp
from jax import jit, lax
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)  # float64 for mass conservation
print(f"JAX {jax.__version__}, devices: {jax.devices()}")

JAX 0.10.0, devices: [CpuDevice(id=0)]


In [3]:
# D2Q9 lattice — Python lists for static values, JAX array for weights only
w  = jnp.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36])
ex = [0, 1, 0, -1,  0, 1, -1, -1,  1]
ey = [0, 0, 1,  0, -1, 1,  1, -1, -1]
opp = [0, 3, 4, 1, 2, 7, 8, 5, 6]
cs2 = 1.0 / 3.0

# JAX arrays only needed for momentum sums (einsum-style)
ex_jnp = jnp.array(ex, dtype=jnp.float64)
ey_jnp = jnp.array(ey, dtype=jnp.float64)

In [ ]:
# Domain: 200 wide × 600 tall
Nx, Ny = 200, 600

# Droplet: 100 μm diameter → R = 50 lu (if dx = 1 μm)
R = 50.0

# Phase-field interface
W = 3.0                         # interface width (lattice units)
sigma = 0.01                    # surface tension (lattice units)
beta  = 3.0 * sigma / W
kappa = 6.0 * sigma * W

# Equal-density, equal-viscosity (simplest stable case)
rho0 = 1.0
nu   = 1.0 / 6.0               # kinematic viscosity
tau_f = 3.0 * nu + 0.5         # = 1.0

# Cahn-Hilliard mobility
Gamma_ch = 1.0
M_ch     = 0.01

# --- Physical-to-lattice pressure conversion ---
dx_phys = 1e-6
nu_phys = 7.68e-7
rho_phys = 1614.0
dt_phys = nu * dx_phys**2 / nu_phys
conv_p = rho_phys * (dx_phys / dt_phys)**2

# Use a safe lattice pressure that gives Ma < 0.1
drho = 0.001
rho_in  = rho0 + drho / 2.0
rho_out = rho0 - drho / 2.0

Nx_eff = Nx - 2
u_max_est = Nx_eff**2 * drho * cs2 / (8.0 * nu * Ny)
print(f"Domain: {Nx}×{Ny} ({Nx*1}×{Ny*1} μm)")
print(f"Δρ={drho}, estimated Poiseuille u_max ≈ {u_max_est:.4f} lu (Ma≈{u_max_est/0.577:.4f})")
print(f"3 droplets, R={R:.0f} lu, diameter={2*R:.0f} μm")

In [ ]:
x = jnp.arange(Nx, dtype=jnp.float64)
y = jnp.arange(Ny, dtype=jnp.float64)
X, Y = jnp.meshgrid(x, y, indexing='ij')

xc = Nx / 2.0

# 3 droplets in a vertical line, evenly spaced
droplet_centers = [150.0, 300.0, 450.0]  # y positions

phi0 = jnp.ones((Nx, Ny))  # start with oil everywhere
for yc_d in droplet_centers:
    r = jnp.sqrt((X - xc)**2 + (Y - yc_d)**2)
    phi_drop = 0.5 * (1.0 + jnp.tanh((r - R) / (2.0 * W)))
    phi0 = jnp.minimum(phi0, phi_drop)  # carve out each droplet

plt.figure(figsize=(4, 12))
plt.imshow(phi0.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
plt.colorbar(label='φ')
for yc_d in droplet_centers:
    plt.axhline(yc_d, color='k', ls=':', alpha=0.3)
plt.title(f'3 droplets, R={R:.0f} lu')
plt.show()

print(f"phi range: [{phi0.min():.6f}, {phi0.max():.6f}]")
print(f"Droplet centers (y): {droplet_centers}")
print(f"Gap between surfaces: {droplet_centers[1]-droplet_centers[0]-2*R:.0f} lu")

In [26]:
@jit
def compute_laplacian(field):
    lap = jnp.zeros_like(field)
    for i in range(1, 9):
        lap += w[i] * jnp.roll(jnp.roll(field, -ex[i], axis=0), -ey[i], axis=1)
    return (2.0 / cs2) * (lap - (1.0 - w[0]) * field)

@jit
def compute_gradient(field):
    gx = jnp.zeros_like(field)
    gy = jnp.zeros_like(field)
    for i in range(1, 9):
        shifted = jnp.roll(jnp.roll(field, -ex[i], axis=0), -ey[i], axis=1)
        gx += w[i] * ex[i] * shifted
        gy += w[i] * ey[i] * shifted
    return gx / cs2, gy / cs2

@jit
def compute_divergence(Fx, Fy):
    """Conservative isotropic divergence. Sums to zero over periodic domain."""
    div = jnp.zeros_like(Fx)
    for i in range(1, 9):
        sx = jnp.roll(jnp.roll(Fx, -ex[i], axis=0), -ey[i], axis=1)
        sy = jnp.roll(jnp.roll(Fy, -ex[i], axis=0), -ey[i], axis=1)
        div += w[i] * (ex[i] * sx + ey[i] * sy)
    return div / cs2

@jit
def chem_potential(phi, lap_phi):
    return 2.0 * beta * phi * (1.0 - phi) * (1.0 - 2.0 * phi) - kappa * lap_phi

@jit
def feq_fn(rho, ux, uy):
    eu = ux[..., None] * ex_jnp + uy[..., None] * ey_jnp
    usq = ux**2 + uy**2
    return w * rho[..., None] * (1.0 + eu/cs2 + eu**2/(2.0*cs2**2) - usq[..., None]/(2.0*cs2))

@jit
def forcing_guo(ux, uy, Fx, Fy):
    eu = ux[..., None] * ex_jnp + uy[..., None] * ey_jnp
    return (1.0 - 0.5/tau_f) * w * (
        (ex_jnp - ux[..., None]) * Fx[..., None] / cs2
        + (ey_jnp - uy[..., None]) * Fy[..., None] / cs2
        + eu / cs2**2 * (ex_jnp * Fx[..., None] + ey_jnp * Fy[..., None])
    )

@jit
def stream(f):
    return jnp.stack([
        jnp.roll(jnp.roll(f[..., i], ex[i], axis=0), ey[i], axis=1)
        for i in range(9)
    ], axis=-1)

In [ ]:
# === Wall and Boundary Conditions ===
# Left/Right: bounce-back walls (no-slip)
# Top (y=Ny-1): inlet, pressure rho_in, phi=1 (oil enters)
# Bottom (y=0): outlet, pressure rho_out, phi=zero-gradient (anything can exit)

wall = jnp.zeros((Nx, Ny), dtype=bool)
wall = wall.at[0, :].set(True)     # left wall
wall = wall.at[-1, :].set(True)    # right wall

fluid = ~wall
# Interior = fluid nodes excluding inlet/outlet rows (for CH update)
interior = fluid & (jnp.arange(Ny)[None, :] > 0) & (jnp.arange(Ny)[None, :] < Ny - 1)

opp_jnp = jnp.array(opp)

@jit
def apply_bounce_back(f):
    f_bounced = f[:, :, opp_jnp]
    return jnp.where(wall[..., None], f_bounced, f)

@jit
def apply_phi_walls(phi):
    phi = jnp.where(wall, 1.0, phi)        # side walls: oil-wet
    phi = phi.at[:, -1].set(1.0)            # inlet (top): pure oil
    phi = phi.at[:, 0].set(phi[:, 1])       # outlet (bottom): zero-gradient
    return phi

@jit
def zou_he_inlet(f, rho_target):
    """Zou-He pressure BC at y=Ny-1 (top). Flow enters downward (uy<0).
    Unknown: f4, f7, f8. Known: f0, f1, f2, f3, f5, f6.
    uy = -1 + (f0+f1+f3 + 2*(f2+f5+f6)) / rho
    """
    rho_t = rho_target
    f_top = f[1:-1, -1, :]
    uy_t = -1.0 + (f_top[:, 0] + f_top[:, 1] + f_top[:, 3]
                    + 2.0 * (f_top[:, 2] + f_top[:, 5] + f_top[:, 6])) / rho_t

    f4 = f_top[:, 2] - (2.0 / 3.0) * rho_t * uy_t
    f7 = f_top[:, 5] + 0.5 * (f_top[:, 1] - f_top[:, 3]) - (1.0 / 6.0) * rho_t * uy_t
    f8 = f_top[:, 6] - 0.5 * (f_top[:, 1] - f_top[:, 3]) - (1.0 / 6.0) * rho_t * uy_t

    f = f.at[1:-1, -1, 4].set(f4)
    f = f.at[1:-1, -1, 7].set(f7)
    f = f.at[1:-1, -1, 8].set(f8)
    return f

@jit
def zou_he_outlet(f, rho_target):
    """Zou-He pressure BC at y=0 (bottom). Flow exits downward (uy<0).
    Unknown: f2, f5, f6. Known: f0, f1, f3, f4, f7, f8.
    uy = 1 - (f0+f1+f3 + 2*(f4+f7+f8)) / rho
    """
    rho_t = rho_target
    f_bot = f[1:-1, 0, :]
    uy_t = 1.0 - (f_bot[:, 0] + f_bot[:, 1] + f_bot[:, 3]
                   + 2.0 * (f_bot[:, 4] + f_bot[:, 7] + f_bot[:, 8])) / rho_t

    f2 = f_bot[:, 4] + (2.0 / 3.0) * rho_t * uy_t
    f5 = f_bot[:, 7] - 0.5 * (f_bot[:, 1] - f_bot[:, 3]) + (1.0 / 6.0) * rho_t * uy_t
    f6 = f_bot[:, 8] + 0.5 * (f_bot[:, 1] - f_bot[:, 3]) + (1.0 / 6.0) * rho_t * uy_t

    f = f.at[1:-1, 0, 2].set(f2)
    f = f.at[1:-1, 0, 5].set(f5)
    f = f.at[1:-1, 0, 6].set(f6)
    return f

print(f"Channel: left/right walls, top=inlet, bottom=outlet")
print(f"Outlet phi: zero-gradient (droplet can exit)")
print(f"Interior fluid nodes: {interior.sum():.0f}")
print(f"rho_in={rho_in:.4f}, rho_out={rho_out:.4f}, Δρ={drho:.4f}")

In [ ]:
@jit
def step(state):
    f, phi = state

    phi = apply_phi_walls(phi)
    phi = jnp.clip(phi, 0.0, 1.0)

    # Phase field operators
    lap_phi = compute_laplacian(phi)
    gx, gy  = compute_gradient(phi)
    mu      = chem_potential(phi, lap_phi)

    Fx = mu * gx
    Fy = mu * gy

    # Zero forces at inlet/outlet rows — stencils wrap periodically in y,
    # so Laplacian/gradient at y=0 sees y=Ny-1 (garbage). No surface tension there.
    Fx = Fx.at[:, 0].set(0.0).at[:, -1].set(0.0)
    Fy = Fy.at[:, 0].set(0.0).at[:, -1].set(0.0)

    # Velocity from LBM
    rho = jnp.sum(f, axis=-1)
    ux = (jnp.sum(f * ex_jnp, axis=-1) + 0.5 * Fx) / rho
    uy = (jnp.sum(f * ey_jnp, axis=-1) + 0.5 * Fy) / rho

    ux = jnp.where(wall, 0.0, ux)
    uy = jnp.where(wall, 0.0, uy)

    # LBM collision — fluid nodes only
    feq = feq_fn(rho, ux, uy)
    Fi  = forcing_guo(ux, uy, Fx, Fy)
    f_collided = f - (f - feq) / tau_f + Fi
    f = jnp.where(fluid[..., None], f_collided, f)

    # Stream
    f = stream(f)

    # Boundary conditions (after streaming)
    f = apply_bounce_back(f)
    f = zou_he_inlet(f, rho_in)
    f = zou_he_outlet(f, rho_out)

    # CH update — interior only (skip walls + inlet/outlet rows)
    lap_mu   = compute_laplacian(mu)
    div_flux = compute_divergence(phi * ux, phi * uy)
    ch_update = -div_flux + M_ch * lap_mu
    phi = phi + jnp.where(interior, ch_update, 0.0)

    phi = apply_phi_walls(phi)
    phi = jnp.clip(phi, 0.0, 1.0)
    return (f, phi)

In [ ]:
rho_init = jnp.ones((Nx, Ny)) * rho0
ux0 = jnp.zeros((Nx, Ny))
uy0 = jnp.zeros((Nx, Ny))

f0 = feq_fn(rho_init, ux0, uy0)
phi0_box = apply_phi_walls(phi0)

print(f"3-droplet channel flow: {Nx}×{Ny}, inlet(top) → outlet(bottom)")
print(f"Δρ={drho}, expected Poiseuille u_max ≈ {u_max_est:.4f} lu")

state = (f0, phi0_box)
state = step(state)
print("JIT compiled.")

state = (f0, phi0_box)
chunk_size = 1000
n_chunks = 30  # 30k total steps

def scan_body(state, _):
    return step(state), None

import time
t0 = time.time()

print(f"\n{'step':>6} | {'phi_min':>10} {'phi_max':>10} | {'rho_min':>10} {'rho_max':>10} | {'max|u|':>10} | {'n_drops':>7} | {'phi@y=0':>10}")
print("-" * 100)

for chunk in range(n_chunks):
    state, _ = lax.scan(scan_body, state, None, length=chunk_size)
    f_c, phi_c = state
    f_c.block_until_ready()

    step_num = (chunk + 1) * chunk_size
    rho_c = jnp.sum(f_c, axis=-1)
    max_vel = jnp.max(jnp.sqrt((jnp.sum(f_c * ex_jnp, axis=-1) / rho_c)**2
                                + (jnp.sum(f_c * ey_jnp, axis=-1) / rho_c)**2))

    mask_c = (phi_c < 0.5).astype(jnp.float64) * interior
    n_droplet_cells = mask_c.sum()

    phi_at_outlet = phi_c[Nx//2, 0]

    print(f"{step_num:6d} | {phi_c.min():10.6f} {phi_c.max():10.6f} | {rho_c.min():10.6f} {rho_c.max():10.6f} | {max_vel:10.2e} | {n_droplet_cells:7.0f} | {phi_at_outlet:10.6f}")

    if jnp.isnan(phi_c).any() or jnp.isnan(f_c).any():
        print(f"  *** NaN detected at step {step_num}! ***")
        break

t1 = time.time()
f_final, phi_final = state
print(f"\nDone in {t1-t0:.1f}s ({n_chunks*chunk_size/(t1-t0):.0f} steps/s)")

In [ ]:
rho_final = jnp.sum(f_final, axis=-1)
ux_final = jnp.sum(f_final * ex_jnp, axis=-1) / rho_final
uy_final = jnp.sum(f_final * ey_jnp, axis=-1) / rho_final
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)

mask = (phi_final < 0.5).astype(jnp.float64) * interior
n_droplet_cells = mask.sum()

print("=" * 50)
print(f"3-DROPLET CHANNEL — {n_chunks*chunk_size} STEPS")
print("=" * 50)
print(f"Max velocity:       {vel_mag.max():.4e} lu")
print(f"Droplet cells:      {n_droplet_cells:.0f}")
print(f"phi range:          [{phi_final.min():.6f}, {phi_final.max():.6f}]")
print(f"rho range:          [{rho_final.min():.6f}, {rho_final.max():.6f}]")

In [ ]:
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)

fig, axes = plt.subplots(1, 4, figsize=(16, 12))

im0 = axes[0].imshow(phi_final.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
axes[0].set_title('φ (oil=1, droplet=0)')
plt.colorbar(im0, ax=axes[0], shrink=0.5)

im1 = axes[1].imshow(rho_final.T, origin='lower', cmap='viridis')
axes[1].set_title('ρ (pressure)')
plt.colorbar(im1, ax=axes[1], shrink=0.5)

im2 = axes[2].imshow(uy_final.T, origin='lower', cmap='coolwarm',
                       vmin=-vel_mag.max(), vmax=vel_mag.max())
axes[2].set_title('u_y (flow direction)')
plt.colorbar(im2, ax=axes[2], shrink=0.5)

im3 = axes[3].imshow(vel_mag.T, origin='lower', cmap='hot')
axes[3].set_title(f'|u| (max={vel_mag.max():.2e})')
plt.colorbar(im3, ax=axes[3], shrink=0.5)

for ax in axes:
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()